# Importing Libraries

In [1]:
!pip install torch transformers peft datasets

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import pandas as pd

In [3]:
#Loading Dataset
dataset = pd.read_csv("/kaggle/input/datasets/vidhyashinde/imdb-dataset-project/IMDB_Dataset.csv")

In [4]:
dataset.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
dataset.shape

(50000, 2)

In [6]:
dataset['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
imdb_data = dataset.copy()

In [8]:
#Converting sentiment labels : positive to 1 and negative to 0
imdb_data['sentiment']=pd.get_dummies(imdb_data['sentiment'],dtype=int,drop_first=True)

In [9]:
#Train-Test Split - stratify to maintain the balance of the reviews in both train and test dataset
#X_train, X_test, y_train, y_test = train_test_split(imdb_data['review'],imdb_data['sentiment'],random_state=42,test_size=0.2,stratify=imdb_data['sentiment'])

# Preprocessing data

In [10]:
#Converting the dataframe to HuggingFace Dataset
imdb_dataset = Dataset.from_pandas(imdb_data)

In [11]:
#imdb_dataset[0]
imdb_dataset

Dataset({
    features: ['review', 'sentiment'],
    num_rows: 50000
})

In [12]:
#Splitting dataset into training and testing
imdb_split_dataset = imdb_dataset.train_test_split(test_size=0.1,seed=42)

In [13]:
imdb_split_dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 45000
    })
    test: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 5000
    })
})

In [14]:
#imdb_split_dataset['test'][0:5]

In [15]:
torch.manual_seed(42)

# Loading a pretrained model and its corresponding tokenizer

In [16]:
#Loading a pretrained model and its corresponding tokenizer
model_id = "google/bert_uncased_L-2_H-128_A-2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=2)

config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Perform baseline prediction (before fine-tuning) on a few reviews

In [17]:
#Selecting first five reviews from test dataset
first_five_test_data = imdb_split_dataset['test'][0:5]

In [18]:
first_five_test_data['sentiment']

[1, 1, 0, 1, 0]

In [19]:
len(first_five_test_data['review'])

5

In [20]:
def predict_sentiment(texts, model, tokenizer):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True , max_length=512)
    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs).logits
    preds = torch.argmax(outputs, dim = 1)
    return preds

texts = first_five_test_data['review']

baseline_preds = predict_sentiment(texts, model, tokenizer)

print("Baseline Predictions:", baseline_preds)
actual_prediction = first_five_test_data['sentiment'][0:5]
print("Actual Predictions:" , actual_prediction)

Baseline Predictions: tensor([1, 1, 0, 1, 1])
Actual Predictions: [1, 1, 0, 1, 0]


# Configuring LoRA

## LoRA Configuration

In [21]:
lora_config = LoraConfig(
    task_type = "SEQ_CLS",
    r = 4,
    #r = 8,
    lora_alpha = 16,
    #lora_alpha = 32,
    target_modules = ["query","value"],
    lora_dropout = 0.1
)

In [22]:
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters() # View the small percentage of updated weights

trainable params: 4,354 || all params: 4,390,532 || trainable%: 0.0992


## Tokenizing data

In [23]:
def tokenize_data(sample_data):
    return tokenizer(sample_data["review"], padding=True, truncation=True , max_length=512)

In [24]:
train_dataset = imdb_split_dataset['train']
test_dataset = imdb_split_dataset['test']

tokenized_train_dataset = train_dataset.map(tokenize_data, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_data, batched=True)

tokenized_train_dataset = tokenized_train_dataset.rename_column("sentiment", "labels")
tokenized_test_dataset = tokenized_test_dataset.rename_column("sentiment", "labels")

tokenized_train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [25]:
tokenized_train_dataset

Dataset({
    features: ['review', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 45000
})

## Training Arguments

In [26]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate = 2e-4,
    per_device_train_batch_size = 4,
    num_train_epochs = 1,
    weight_decay = 0.1,
    logging_steps = 10
    #label_names=['sentiment'], # As the default points to label or labels we need to change it to sentiment in our case
    #remove_unused_columns = False # Changed to True to resolve padding conflict
)

In [27]:
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    processing_class=tokenizer
)

In [28]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,1.387612
20,1.384344
30,1.397207
40,1.378388
50,1.376589
60,1.361819
70,1.381025
80,1.419858
90,1.361232
100,1.351358


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=5625, training_loss=0.9977124139997694, metrics={'train_runtime': 167.7755, 'train_samples_per_second': 268.216, 'train_steps_per_second': 33.527, 'total_flos': 57773813760000.0, 'train_loss': 0.9977124139997694, 'epoch': 1.0})

In [29]:
model.save_pretrained("./lora-sentiment-model")
print("Model trained and saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model trained and saved successfully.


In [30]:
tokenizer.save_pretrained("./lora-sentiment-model")

('./lora-sentiment-model/tokenizer_config.json',
 './lora-sentiment-model/tokenizer.json')

In [31]:
# device = torch.device("cuda")
# model.cuda()

In [33]:
finetuned_preds = predict_sentiment(texts, model, tokenizer)
print(finetuned_preds)

tensor([1, 1, 0, 1, 0], device='cuda:0')


In [36]:
sentiment_ds=pd.DataFrame({
    'Actual Prediction': actual_prediction,
    'Baseline Predictions': baseline_preds.cpu().tolist(),
    'Finetuned Predictions': finetuned_preds.cpu().tolist()
})
print(sentiment_ds)

   Actual Prediction  Baseline Predictions  Finetuned Predictions
0                  1                     1                      1
1                  1                     1                      1
2                  0                     0                      0
3                  1                     1                      1
4                  0                     1                      0
